In [ ]:
!pip -q install feast==0.64.0 pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires tenacity<10,>

In [ ]:
import feast
import pyarrow

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import feast

print("Feast version:", feast.__version__)

Feast version: 0.64.0


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Curriculum_Industry_Skill_Alignment_Dataset.xlsx to Curriculum_Industry_Skill_Alignment_Dataset.xlsx


In [ ]:
import pandas as pd

df = pd.read_excel(
    "Curriculum_Industry_Skill_Alignment_Dataset.xlsx",
    sheet_name="Student_Skill_Data"
)

print(df.head())
print("Dataset shape:", df.shape)

  Student_ID  Programming  Database  Problem_Solving  Communication  \
0     CSE001           75        77               57             71   
1     CSE002           66        51               66             85   
2     CSE003           77        86               74             62   
3     CSE004           89        44               78             70   
4     CSE005           65        74               51             74   

   Cloud_Computing  Data_Analysis  Teamwork  Aptitude  Programming_Gap  ...  \
0               53             70        74        65                5  ...   
1               39             43        49        75               14  ...   
2               28             72        54        50                3  ...   
3               58             80        79        71                0  ...   
4               42             65        72        68               15  ...   

   Problem_Solving_Gap  Communication_Gap  Cloud_Computing_Gap  \
0                   23          

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

Columns:
['Student_ID', 'Programming', 'Database', 'Problem_Solving', 'Communication', 'Cloud_Computing', 'Data_Analysis', 'Teamwork', 'Aptitude', 'Programming_Gap', 'Database_Gap', 'Problem_Solving_Gap', 'Communication_Gap', 'Cloud_Computing_Gap', 'Data_Analysis_Gap', 'Teamwork_Gap', 'Aptitude_Gap', 'Average_Skill_Score', 'Average_Skill_Gap', 'Skill_Gap_Category', 'Recommended_Training_Area']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Student_ID                 120 non-null    object 
 1   Programming                120 non-null    int64  
 2   Database                   120 non-null    int64  
 3   Problem_Solving            120 non-null    int64  
 4   Communication              120 non-null    int64  
 5   Cloud_Computing            120 non-null    int64  
 6   Data_Analysis          

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import pandas as pd

# Make a copy
df = df.copy()

# Entity
df["student_id"] = df["Student_ID"].astype(str)

# CSE technical and employability skills
skill_columns = [
    "Programming",
    "Database",
    "Problem_Solving",
    "Communication",
    "Cloud_Computing",
    "Data_Analysis",
    "Teamwork",
    "Aptitude"
]

# Convert skill columns to numeric
for col in skill_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing skill values with median
for col in skill_columns:
    df[col] = df[col].fillna(df[col].median())

# Create technical skill score
technical_skills = [
    "Programming",
    "Database",
    "Cloud_Computing",
    "Data_Analysis"
]

df["technical_skill_score"] = (
    df[technical_skills].mean(axis=1)
).round(2)

# Create professional/employability skill score
professional_skills = [
    "Problem_Solving",
    "Communication",
    "Teamwork",
    "Aptitude"
]

df["professional_skill_score"] = (
    df[professional_skills].mean(axis=1)
).round(2)

# Overall calculated skill score
df["calculated_skill_score"] = (
    df[skill_columns].mean(axis=1)
).round(2)

# Existing dataset target
df["average_skill_score"] = pd.to_numeric(
    df["Average_Skill_Score"],
    errors="coerce"
)

df["average_skill_gap"] = pd.to_numeric(
    df["Average_Skill_Gap"],
    errors="coerce"
)

# Skill-gap category
df["skill_gap_category"] = (
    df["Skill_Gap_Category"].astype(str)
)

# Recommended training area
df["recommended_training_area"] = (
    df["Recommended_Training_Area"].astype(str)
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
display(
    df[
        [
            "student_id",
            "Programming",
            "Database",
            "Problem_Solving",
            "Communication",
            "Cloud_Computing",
            "Data_Analysis",
            "Teamwork",
            "Aptitude",
            "technical_skill_score",
            "professional_skill_score",
            "calculated_skill_score",
            "average_skill_gap",
            "skill_gap_category"
        ]
    ].head()
)

,student_id,Programming,Database,Problem_Solving,Communication,Cloud_Computing,Data_Analysis,Teamwork,Aptitude,technical_skill_score,professional_skill_score,calculated_skill_score,average_skill_gap,skill_gap_category
0,CSE001,75,77,57,71,53,70,74,65,68.75,66.75,67.75,7.50,Low Gap
1,CSE002,66,51,66,85,39,43,49,75,49.75,68.75,59.25,17.00,Medium Gap
2,CSE003,77,86,74,62,28,72,54,50,65.75,60.00,62.88,13.75,Medium Gap
3,CSE004,89,44,78,70,58,80,79,71,67.75,74.50,71.12,6.75,Low Gap
4,CSE005,65,74,51,74,42,65,72,68,61.50,66.25,63.88,11.12,Medium Gap


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
base_time = pd.Timestamp(
    "2025-01-01",
    tz="UTC"
)

# Extract numeric part from CSE001, CSE002, etc.
student_number = (
    df["student_id"]
    .str.extract(r"(\d+)", expand=False)
    .astype(int)
)

df["event_timestamp"] = (
    base_time +
    pd.to_timedelta(
        student_number,
        unit="D"
    )
)

df["created_timestamp"] = (
    df["event_timestamp"] +
    pd.Timedelta(hours=1)
)

print(
    df[
        [
            "student_id",
            "event_timestamp",
            "created_timestamp"
        ]
    ].head()
)

  student_id           event_timestamp         created_timestamp
0     CSE001 2025-01-02 00:00:00+00:00 2025-01-02 01:00:00+00:00
1     CSE002 2025-01-03 00:00:00+00:00 2025-01-03 01:00:00+00:00
2     CSE003 2025-01-04 00:00:00+00:00 2025-01-04 01:00:00+00:00
3     CSE004 2025-01-05 00:00:00+00:00 2025-01-05 01:00:00+00:00
4     CSE005 2025-01-06 00:00:00+00:00 2025-01-06 01:00:00+00:00


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
feature_df = df[
    [
        "student_id",
        "event_timestamp",
        "created_timestamp",

        "Programming",
        "Database",
        "Problem_Solving",
        "Communication",
        "Cloud_Computing",
        "Data_Analysis",
        "Teamwork",
        "Aptitude",

        "technical_skill_score",
        "professional_skill_score",
        "calculated_skill_score"
    ]
].copy()

# Convert numeric features to appropriate types
integer_features = [
    "Programming",
    "Database",
    "Problem_Solving",
    "Communication",
    "Cloud_Computing",
    "Data_Analysis",
    "Teamwork",
    "Aptitude"
]

for col in integer_features:
    feature_df[col] = feature_df[col].round().astype("int64")

float_features = [
    "technical_skill_score",
    "professional_skill_score",
    "calculated_skill_score"
]

for col in float_features:
    feature_df[col] = feature_df[col].astype("float32")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
label_df = df[
    [
        "student_id",
        "event_timestamp",
        "skill_gap_category",
        "average_skill_gap",
        "recommended_training_area"
    ]
].copy()

print("Label distribution:")
print(label_df["skill_gap_category"].value_counts())

Label distribution:
skill_gap_category
Medium Gap    88
Low Gap       30
High Gap       2
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
print("Feature data:")
display(feature_df.head())

print("\nLabel data:")
display(label_df.head())

Feature data:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,student_id,event_timestamp,created_timestamp,Programming,Database,Problem_Solving,Communication,Cloud_Computing,Data_Analysis,Teamwork,Aptitude,technical_skill_score,professional_skill_score,calculated_skill_score
0,CSE001,2025-01-02 00:00:00+00:00,2025-01-02 01:00:00+00:00,75,77,57,71,53,70,74,65,68.75,66.75,67.750000
1,CSE002,2025-01-03 00:00:00+00:00,2025-01-03 01:00:00+00:00,66,51,66,85,39,43,49,75,49.75,68.75,59.250000
2,CSE003,2025-01-04 00:00:00+00:00,2025-01-04 01:00:00+00:00,77,86,74,62,28,72,54,50,65.75,60.00,62.880001
3,CSE004,2025-01-05 00:00:00+00:00,2025-01-05 01:00:00+00:00,89,44,78,70,58,80,79,71,67.75,74.50,71.120003
4,CSE005,2025-01-06 00:00:00+00:00,2025-01-06 01:00:00+00:00,65,74,51,74,42,65,72,68,61.50,66.25,63.880001



Label data:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,student_id,event_timestamp,skill_gap_category,average_skill_gap,recommended_training_area
0,CSE001,2025-01-02 00:00:00+00:00,Low Gap,7.50,Problem Solving + Cloud Computing
1,CSE002,2025-01-03 00:00:00+00:00,Medium Gap,17.00,Cloud Computing + Data Analysis
2,CSE003,2025-01-04 00:00:00+00:00,Medium Gap,13.75,Cloud Computing + Aptitude
3,CSE004,2025-01-05 00:00:00+00:00,Low Gap,6.75,Database + Cloud Computing
4,CSE005,2025-01-06 00:00:00+00:00,Medium Gap,11.12,Problem Solving + Cloud Computing


In [ ]:
import os

repo_path = "/content/cse_skill_feast"

os.makedirs(
    f"{repo_path}/data",
    exist_ok=True
)

print("Repository created:", repo_path)

Repository created: /content/cse_skill_feast


In [ ]:
import os

repo_path = "/content/cse_skill_feast"

os.makedirs(
    f"{repo_path}/data",
    exist_ok=True
)

print("Repository ready.")

Repository ready.


In [ ]:
feature_df.to_parquet(
    f"{repo_path}/data/cse_skill_features.parquet",
    index=False
)

print("Feature data saved successfully.")

Feature data saved successfully.


In [ ]:
feature_store_yaml = """
project: cse_skill_project

registry: data/registry.db

provider: local

offline_store:
  type: file

online_store:
  type: sqlite
  path: data/online_store.db
"""

with open(
    f"{repo_path}/feature_store.yaml",
    "w"
) as f:
    f.write(feature_store_yaml)

print("feature_store.yaml created.")

feature_store.yaml created.


In [ ]:
feature_definition = '''
from datetime import timedelta

from feast import (
    Entity,
    FeatureView,
    FeatureService,
    Field,
    FileSource
)

from feast.types import (
    Float32,
    Int64
)


# -----------------------------
# ENTITY
# -----------------------------

student = Entity(
    name="student",
    join_keys=["student_id"],
    description="CSE student"
)


# -----------------------------
# DATA SOURCE
# -----------------------------

cse_skill_source = FileSource(
    name="cse_skill_source",
    path="data/cse_skill_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp"
)


# -----------------------------
# FEATURE VIEW
# -----------------------------

cse_skill_feature_view = FeatureView(
    name="cse_skill_features",
    entities=[student],

    ttl=timedelta(days=3650),

    schema=[
        Field(name="Programming", dtype=Int64),
        Field(name="Database", dtype=Int64),
        Field(name="Problem_Solving", dtype=Int64),
        Field(name="Communication", dtype=Int64),
        Field(name="Cloud_Computing", dtype=Int64),
        Field(name="Data_Analysis", dtype=Int64),
        Field(name="Teamwork", dtype=Int64),
        Field(name="Aptitude", dtype=Int64),

        Field(
            name="technical_skill_score",
            dtype=Float32
        ),

        Field(
            name="professional_skill_score",
            dtype=Float32
        ),

        Field(
            name="calculated_skill_score",
            dtype=Float32
        )
    ],

    source=cse_skill_source,

    online=True
)


# -----------------------------
# FEATURE SERVICE
# -----------------------------

cse_skill_feature_service = FeatureService(
    name="cse_skill_gap_service",
    features=[
        cse_skill_feature_view
    ]
)
'''

with open(
    f"{repo_path}/features.py",
    "w"
) as f:
    f.write(feature_definition)

print("features.py created.")

features.py created.


In [ ]:
!find /content/cse_skill_feast -maxdepth 2 -type f

/content/cse_skill_feast/feature_store.yaml
/content/cse_skill_feast/data/cse_skill_features.parquet
/content/cse_skill_feast/features.py


In [ ]:
%cd /content/cse_skill_feast

/content/cse_skill_feast


In [ ]:
!feast apply

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [ ]:
!feast entities list

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [ ]:
!feast feature-views list

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [ ]:
from feast import FeatureStore

store = FeatureStore(
    repo_path="/content/cse_skill_feast"
)

print("Feast FeatureStore created.")

Feast FeatureStore created.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
feature_service = store.get_feature_service(
    "cse_skill_gap_service"
)

print("Feature service loaded successfully.")

Feature service loaded successfully.


In [ ]:
entity_df = label_df[
    [
        "student_id",
        "event_timestamp",
        "skill_gap_category",
        "average_skill_gap",
        "recommended_training_area"
    ]
].copy()

display(entity_df.head())

,student_id,event_timestamp,skill_gap_category,average_skill_gap,recommended_training_area
0,CSE001,2025-01-02 00:00:00+00:00,Low Gap,7.50,Problem Solving + Cloud Computing
1,CSE002,2025-01-03 00:00:00+00:00,Medium Gap,17.00,Cloud Computing + Data Analysis
2,CSE003,2025-01-04 00:00:00+00:00,Medium Gap,13.75,Cloud Computing + Aptitude
3,CSE004,2025-01-05 00:00:00+00:00,Low Gap,6.75,Database + Cloud Computing
4,CSE005,2025-01-06 00:00:00+00:00,Medium Gap,11.12,Problem Solving + Cloud Computing


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
training_data = store.get_historical_features(
    entity_df=entity_df,
    features=feature_service
).to_df()

print(
    "Historical feature retrieval completed."
)

Historical feature retrieval completed.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
display(
    training_data.head()
)

print(
    "Training data shape:",
    training_data.shape
)

,student_id,event_timestamp,skill_gap_category,average_skill_gap,recommended_training_area,Programming,Database,Problem_Solving,Communication,Cloud_Computing,Data_Analysis,Teamwork,Aptitude,technical_skill_score,professional_skill_score,calculated_skill_score
0,CSE001,2025-01-02 00:00:00+00:00,Low Gap,7.50,Problem Solving + Cloud Computing,75,77,57,71,53,70,74,65,68.75,66.75,67.750000
1,CSE002,2025-01-03 00:00:00+00:00,Medium Gap,17.00,Cloud Computing + Data Analysis,66,51,66,85,39,43,49,75,49.75,68.75,59.250000
2,CSE003,2025-01-04 00:00:00+00:00,Medium Gap,13.75,Cloud Computing + Aptitude,77,86,74,62,28,72,54,50,65.75,60.00,62.880001
3,CSE004,2025-01-05 00:00:00+00:00,Low Gap,6.75,Database + Cloud Computing,89,44,78,70,58,80,79,71,67.75,74.50,71.120003
4,CSE005,2025-01-06 00:00:00+00:00,Medium Gap,11.12,Problem Solving + Cloud Computing,65,74,51,74,42,65,72,68,61.50,66.25,63.880001


Training data shape: (120, 16)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
feature_columns = [
    "Programming",
    "Database",
    "Problem_Solving",
    "Communication",
    "Cloud_Computing",
    "Data_Analysis",
    "Teamwork",
    "Aptitude"
]

print("ML Features:")
print(feature_columns)

ML Features:
['Programming', 'Database', 'Problem_Solving', 'Communication', 'Cloud_Computing', 'Data_Analysis', 'Teamwork', 'Aptitude']


In [ ]:
X = training_data[
    feature_columns
]

y = training_data[
    "skill_gap_category"
]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget classes:")
print(y.value_counts())

X shape: (120, 8)
y shape: (120,)

Target classes:
skill_gap_category
Medium Gap    88
Low Gap       30
High Gap       2
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 96
Testing samples: 24


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

print("Decision Tree model trained successfully.")

Decision Tree model trained successfully.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
predictions = model.predict(
    X_test
)

print("Predictions:")
print(predictions[:20])

Predictions:
['Medium Gap' 'Medium Gap' 'Medium Gap' 'Low Gap' 'Low Gap' 'Medium Gap'
 'Low Gap' 'Low Gap' 'Medium Gap' 'Medium Gap' 'Medium Gap' 'Medium Gap'
 'Low Gap' 'Medium Gap' 'Medium Gap' 'Medium Gap' 'Medium Gap' 'Low Gap'
 'Low Gap' 'Low Gap']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

accuracy = accuracy_score(
    y_test,
    predictions
)

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions
    )
)

Accuracy: 75.0 %

Classification Report:
              precision    recall  f1-score   support

     Low Gap       0.50      0.67      0.57         6
  Medium Gap       0.88      0.78      0.82        18

    accuracy                           0.75        24
   macro avg       0.69      0.72      0.70        24
weighted avg       0.78      0.75      0.76        24



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
%cd /content/cse_skill_feast

!feast materialize \
    2025-01-01T00:00:00 \
    2025-05-01T00:00:00

/content/cse_skill_feast
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib

In [ ]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {
            "student_id": "CSE025"
        }
    ]
).to_dict()

print("Online features retrieved successfully.")

Online features retrieved successfully.


In [ ]:
print(online_features)

{'student_id': ['CSE025'], 'Database': [69], 'Programming': [60], 'Teamwork': [76], 'professional_skill_score': [62.5], 'Data_Analysis': [83], 'Cloud_Computing': [44], 'calculated_skill_score': [63.25], 'Problem_Solving': [64], 'Aptitude': [57], 'technical_skill_score': [64.0], 'Communication': [53]}


In [ ]:
online_df = pd.DataFrame(
    online_features
)

display(
    online_df
)

,student_id,Database,Programming,Teamwork,professional_skill_score,Data_Analysis,Cloud_Computing,calculated_skill_score,Problem_Solving,Aptitude,technical_skill_score,Communication
0,CSE025,69,60,76,62.5,83,44,63.25,64,57,64.0,53


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
final_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

final_model.fit(
    X,
    y
)

print("Final model trained on all available historical features.")

Final model trained on all available historical features.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
X_online = online_df[
    feature_columns
]

display(
    X_online
)

,Programming,Database,Problem_Solving,Communication,Cloud_Computing,Data_Analysis,Teamwork,Aptitude
0,60,69,64,53,44,83,76,57


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
prediction = final_model.predict(
    X_online
)

print(
    "Student:",
    online_df["student_id"].iloc[0]
)

print(
    "Predicted Skill Gap:",
    prediction[0]
)

Student: CSE025
Predicted Skill Gap: Medium Gap


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {"student_id": "CSE010"},
        {"student_id": "CSE020"},
        {"student_id": "CSE030"},
        {"student_id": "CSE040"},
        {"student_id": "CSE050"}
    ]
).to_dict()

online_df = pd.DataFrame(
    online_features
)

display(
    online_df
)

,student_id,Database,Programming,Teamwork,professional_skill_score,Data_Analysis,Cloud_Computing,calculated_skill_score,Problem_Solving,Aptitude,technical_skill_score,Communication
0,CSE010,57,76,61,61.25,82,70,66.250000,72,59,71.25,53
1,CSE020,47,48,82,71.75,62,31,59.380001,77,75,47.00,53
2,CSE030,69,64,53,68.00,76,41,65.250000,69,94,62.50,56
3,CSE040,75,71,78,76.00,37,68,69.379997,62,83,62.75,81
4,CSE050,54,43,92,76.00,69,40,63.750000,59,82,51.50,71


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
online_df["predicted_skill_gap"] = (
    final_model.predict(
        online_df[feature_columns]
    )
)

result_df = online_df[
    [
        "student_id",
        "predicted_skill_gap"
    ]
].copy()

display(
    result_df
)

,student_id,predicted_skill_gap
0,CSE010,Medium Gap
1,CSE020,Medium Gap
2,CSE030,Medium Gap
3,CSE040,Low Gap
4,CSE050,Medium Gap


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
